# EEG_08c — Subject Quality Metrics (EEG-First)

Calcola metriche EEG-first per identificare soggetti 'buoni' senza usare accuracy.

| Metrica | Descrizione |
|---------|-------------|
| **Inter-session consistency** | Correlazione matrici adj tra sessioni |
| **Band power variance** | Varianza band power alpha/beta tra sessioni |
| **Intra-class coherence** | Similarita trial della stessa parola |

Finale: correlazione con ranking EEG_08b.

## 1 — Imports & Config

In [ ]:
import json
import logging
import re
from collections import defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import signal as scipy_signal
from scipy.stats import pearsonr, spearmanr
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s  %(levelname)-8s  %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('eeg08c')

project_root = next(
    (p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / '.git').exists()),
    Path.cwd(),
)
(project_root / 'figures').mkdir(exist_ok=True)

SFREQ       = 256          # Hz
N_CHANNELS  = 61
CSV_ROOT    = project_root / 'data' / 'raw_csv' / 'training_set'
PT_ROOT     = project_root / 'data' / 'graphs_abs_pcc'

# bande frequenza
BANDS = {
    'delta': (1,  4),
    'theta': (4,  8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 45),
}

_PAT = re.compile(r'^P(\d+)_S(\d+)$')

# raccolta path per soggetto -> sessione (dai .pt)
subj_sess_pts = defaultdict(lambda: defaultdict(list))
for p in sorted(PT_ROOT.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if m:
        subj_sess_pts[int(m.group(1))][int(m.group(2))].append(p)

# raccolta path CSV per soggetto -> sessione
subj_sess_csv = defaultdict(lambda: defaultdict(list))
for sess_dir in sorted(CSV_ROOT.iterdir()):
    m = _PAT.match(sess_dir.name)
    if m:
        subj_sess_csv[int(m.group(1))][int(m.group(2))].extend(
            sorted(sess_dir.glob('*_img.csv')))

ALL_SUBJ_IDS = sorted(set(subj_sess_pts.keys()) & set(subj_sess_csv.keys()))
log.info(f'Soggetti: {len(ALL_SUBJ_IDS)}')


## 2 — Inter-Session Connectivity Consistency

Per ogni soggetto: carica mean adj per sessione dai .pt, calcola
correlazione di Pearson tra tutte le coppie di sessioni.

Alta correlazione = topologia di connettivita stabile nel tempo.

In [ ]:
def mean_adj_for_session(pt_paths):
    adjs = []
    for p in pt_paths:
        d = torch.load(p, weights_only=False)
        adjs.append(d['adj'].numpy() if isinstance(d['adj'], torch.Tensor)
                    else np.array(d['adj']))
    if not adjs: return None
    return np.mean(adjs, axis=0)


def inter_session_consistency(subj_id):
    sess = subj_sess_pts[subj_id]
    sids = sorted(sess.keys())
    if len(sids) < 2: return np.nan
    means = [mean_adj_for_session(sess[s]) for s in sids]
    means = [m for m in means if m is not None]
    if len(means) < 2: return np.nan
    corrs = []
    for a, b in combinations(means, 2):
        # upper triangle, escludi diagonale
        idx = np.triu_indices(a.shape[0], k=1)
        r, _ = pearsonr(a[idx], b[idx])
        corrs.append(r)
    return float(np.mean(corrs))


log.info('Calcolo inter-session consistency...')
isc_scores = {}
for sid in tqdm(ALL_SUBJ_IDS, desc='ISC'):
    isc_scores[sid] = inter_session_consistency(sid)

log.info(f'ISC mean={np.nanmean(list(isc_scores.values())):.4f}  '
         f'std={np.nanstd(list(isc_scores.values())):.4f}')


## 3 — Band Power Variance Inter-Sessione

Per ogni soggetto e banda: calcola band power media per sessione,
poi varianza tra sessioni.

Bassa varianza = risposta spettrale stabile = soggetto piu affidabile.

In [ ]:
def bandpower_trial(x, sfreq, fmin, fmax):
    """Band power media su tutti i canali per un trial (61, T)."""
    nperseg = min(256, x.shape[1])
    freqs, psd = scipy_signal.welch(x, fs=sfreq, nperseg=nperseg, axis=1)
    idx = np.logical_and(freqs >= fmin, freqs <= fmax)
    return float(np.mean(psd[:, idx]))


def session_bandpower(csv_paths, band_range):
    """Media band power su tutti i trial di una sessione."""
    bps = []
    for p in csv_paths[:20]:  # max 20 trial per velocita
        try:
            x = pd.read_csv(p, header=None).values.astype(np.float32)
            if x.shape != (N_CHANNELS, 384): continue
            bps.append(bandpower_trial(x, SFREQ, *band_range))
        except Exception:
            continue
    return float(np.mean(bps)) if bps else np.nan


log.info('Calcolo band power variance...')
bp_var_scores = defaultdict(dict)  # subj_id -> band -> variance

for sid in tqdm(ALL_SUBJ_IDS, desc='BandPower'):
    sess = subj_sess_csv[sid]
    sids = sorted(sess.keys())
    for band, (fmin, fmax) in BANDS.items():
        sess_bps = [session_bandpower(sess[s], (fmin, fmax)) for s in sids]
        sess_bps = [v for v in sess_bps if not np.isnan(v)]
        bp_var_scores[sid][band] = float(np.var(sess_bps)) if len(sess_bps) >= 2 else np.nan

log.info('Band power variance OK')


## 4 — Intra-Class Coherence

Per ogni soggetto: per ogni parola, calcola correlazione tra adj di trial
diversi della stessa parola (cross-session).

Alta coerenza = il cervello risponde in modo simile alla stessa parola.

In [ ]:
_raw_map = json.loads(
    (project_root / 'configs' / 'label_schemes' / 'labelid2cluster_concr4.json').read_text()
)
LABEL2CLUSTER_NP = {int(k): int(v) for k, v in _raw_map.items()}


def intra_class_coherence(subj_id, max_pairs=100):
    """Correlazione media tra adj di trial della stessa classe."""
    all_pts = [p for sess in subj_sess_pts[subj_id].values() for p in sess]
    # raggruppa per cluster
    cluster_adjs = defaultdict(list)
    for p in all_pts:
        d = torch.load(p, weights_only=False)
        y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
        c = LABEL2CLUSTER_NP.get(y_word)
        if c is not None:
            adj = d['adj'].numpy() if isinstance(d['adj'], torch.Tensor) else np.array(d['adj'])
            cluster_adjs[c].append(adj)

    corrs = []
    for c, adjs in cluster_adjs.items():
        if len(adjs) < 2: continue
        pairs = list(combinations(range(len(adjs)), 2))
        if len(pairs) > max_pairs:
            pairs = pairs[:max_pairs]
        idx = np.triu_indices(adjs[0].shape[0], k=1)
        for i, j in pairs:
            r, _ = pearsonr(adjs[i][idx], adjs[j][idx])
            corrs.append(r)
    return float(np.mean(corrs)) if corrs else np.nan


log.info('Calcolo intra-class coherence...')
icc_scores = {}
for sid in tqdm(ALL_SUBJ_IDS, desc='ICC'):
    icc_scores[sid] = intra_class_coherence(sid)

log.info(f'ICC mean={np.nanmean(list(icc_scores.values())):.4f}')


## 5 — Correlazione con Ranking EEG_08b

In [ ]:
# carica ranking EEG_08b
ranking_path = project_root / 'figures' / 'eeg08b_subject_ranking.csv'
assert ranking_path.exists(), 'Riesegui EEG_08b prima'
df_08b = pd.read_csv(ranking_path, index_col=0)
df_08b['subj_id'] = df_08b['Subject'].str[1:].astype(int)

# assembla dataframe metriche
rows = []
for sid in ALL_SUBJ_IDS:
    if sid not in {r for r in df_08b['subj_id']}:
        continue
    row = {
        'subj_id':  sid,
        'isc':      isc_scores.get(sid, np.nan),
        'icc':      icc_scores.get(sid, np.nan),
    }
    for band in BANDS:
        row[f'bp_var_{band}'] = bp_var_scores[sid].get(band, np.nan)
    rows.append(row)

df_metrics = pd.DataFrame(rows)
df_merged  = df_08b.merge(df_metrics, on='subj_id')

# correlazioni Spearman con test_bacc
metric_cols = ['isc', 'icc'] + [f'bp_var_{b}' for b in BANDS]
print('Correlazione Spearman con Test bAcc (EEG_08b):')
print('-' * 45)
corr_results = []
for col in metric_cols:
    valid = df_merged[['Test bAcc', col]].dropna()
    if len(valid) < 5: continue
    r, p = spearmanr(valid['Test bAcc'], valid[col])
    corr_results.append({'Metrica': col, 'Spearman r': round(r, 3), 'p-value': round(p, 4)})
    print(f'  {col:20s}  r={r:+.3f}  p={p:.4f}')

df_corr = pd.DataFrame(corr_results).sort_values('Spearman r', ascending=False)


## 6 — Plot Finale

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('EEG_08c — Subject Quality Metrics vs GCN Accuracy',
             fontsize=13, fontweight='bold')

for ax, (xcol, xlabel) in zip(axes, [
    ('isc',       'Inter-Session Consistency (Pearson r)'),
    ('icc',       'Intra-Class Coherence (Pearson r)'),
    ('bp_var_alpha', 'Alpha Band Power Variance (inter-session)'),
]):
    valid = df_merged[['Test bAcc', xcol, 'Subject']].dropna()
    ax.scatter(valid[xcol], valid['Test bAcc'], alpha=0.6, s=40, color='#4e79a7')
    # trend line
    if len(valid) > 3:
        z = np.polyfit(valid[xcol], valid['Test bAcc'], 1)
        xr = np.linspace(valid[xcol].min(), valid[xcol].max(), 50)
        ax.plot(xr, np.polyval(z, xr), 'r--', lw=1.5)
    r, p = spearmanr(valid['Test bAcc'], valid[xcol])
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Test Balanced Accuracy', fontsize=10)
    ax.set_title(f'r={r:+.3f}  p={p:.3f}', fontsize=10)
    ax.axhline(0.25, color='gray', lw=1, linestyle='--', alpha=0.5)

plt.tight_layout()
fig.savefig(project_root / 'figures' / 'eeg08c_quality_metrics.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(df_corr.to_string(index=False))
df_corr.to_csv(project_root / 'figures' / 'eeg08c_correlations.csv', index=False)
print('Salvato eeg08c_quality_metrics.png + eeg08c_correlations.csv')
